# CSM Strategy — June 2026 Rebalance (2 มิ.ย. 2026 ATO)

**Rebalance Date:** 2 มิถุนายน 2026 (ATO) &nbsp;|&nbsp; **Current Holdings:** 9 หุ้น &nbsp;|&nbsp; **Selected:** Option C — Backtest Top-10

---

*สมุดบันทึกนี้แสดงข้อมูลการ Rebalance จากโมเดล CSM ทั้งหมด — composite ranking, การเลือกหุ้น, sector cap, และเหตุผลที่เลือก Option C ล้วนคำนวณจาก features_latest.parquet ที่ refresh ล่าสุด 1 มิถุนายน 2026*

In [ ]:
import os, sys, warnings, json
from pathlib import Path
from datetime import date

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import yaml

warnings.filterwarnings("ignore")

# ── Style ──────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 120,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.titlesize": 15,
    "figure.facecolor": "white",
    "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc",
    "axes.grid": True,
    "grid.alpha": 0.4,
    "grid.color": "#dddddd",
})
sns.set_palette("viridis")

# ── Resolve project root ───────────────────────────────────
_HERE = Path.cwd()
_PROJECT_ROOT = _HERE.parent if _HERE.name == "notebooks" else _HERE
os.chdir(_PROJECT_ROOT)
sys.path.insert(0, str(_PROJECT_ROOT / "src"))

DATA = _PROJECT_ROOT / "data" / "processed"
CONFIGS = _PROJECT_ROOT / "configs"
print(f"Project root: {_PROJECT_ROOT}")
print(f"Data dir: {DATA}")

# ── Load data from model ───────────────────────────────────
features = pd.read_parquet(DATA / "features_latest.parquet")
prices = pd.read_parquet(DATA / "prices_latest.parquet")
universe = pd.read_parquet(DATA / "universe_latest.parquet")

# Load live portfolio config
with open(CONFIGS / "live_portfolio.yaml") as f:
    live_config = yaml.safe_load(f)

current_holdings = [p["symbol"] for p in live_config["positions"]]
cash = live_config["cash"]
starting_nav = live_config["starting_nav"]

# Features: isolate latest date cross-section
prices.index = pd.to_datetime(prices.index)
latest_date = features["date"].max()
latest = features[features["date"] == latest_date].copy()
latest["ticker"] = latest["symbol"].str.replace("SET:", "")
latest = latest.set_index("ticker")

# Feature columns from the model (what Pipeline actually writes)
FEATURE_COLS = ["mom_12_1", "mom_6_1", "mom_3_1", "mom_1_0"]

# Composite z-score = cross_section.mean(axis=1)  ← exactly PortfolioConstructor.select()
composite = latest[FEATURE_COLS].mean(axis=1)
pct_rank = composite.rank(pct=True)
latest["composite"] = composite
latest["pct_rank"] = pct_rank

# Sort by composite descending
latest = latest.sort_values("composite", ascending=False)

print(f"Data source: features_latest.parquet")
print(f"Latest date: {latest_date}")
print(f"Universe size: {len(latest)} symbols")
print(f"Feature columns: {FEATURE_COLS}")
print(f"Current holdings: {len(current_holdings)} ({', '.join(current_holdings)})")
print(f"Cash: {cash:,.2f} THB")
print("Ready.")

## 1. อันดับ Composite ปัจจุบัน (Current Composite Ranking)

Composite z-score = ค่าเฉลี่ยของ 4 momentum signals (`mom_12_1`, `mom_6_1`, `mom_3_1`, `mom_1_0`) แบบ equal-weight ตาม `PortfolioConstructor.select()` — นี่คือแบบจำลองเดียวกับที่ใช้ใน backtest

Exit Rules:
- **Exit Rank Floor = 0.35** — ขายทันทีถ้า percentile rank ต่ำกว่า 35th
- **Buffer Threshold = 0.25** — เปลี่ยนตัวก็ต่อเมื่อหุ้นใหม่ rank สูงกว่าหุ้นเดิม ≥ 25 percentile points

In [ ]:
# ── Current holdings: composite rank ───────────────────────
EXIT_FLOOR = 0.35
BUFFER = 0.25

held_data = []
for sym in current_holdings:
    row = {
        "Symbol": sym,
        "Composite Z": latest.loc[sym, "composite"],
        "Pct Rank": latest.loc[sym, "pct_rank"],
        "Rank": (composite > latest.loc[sym, "composite"]).sum() + 1,
        "Below 0.35 Floor?": "⚠️ YES" if latest.loc[sym, "pct_rank"] < EXIT_FLOOR else "no",
    }
    held_data.append(row)

held_df = pd.DataFrame(held_data).sort_values("Pct Rank", ascending=False)
held_df = held_df.reset_index(drop=True)

print(f"Exit Rank Floor: {EXIT_FLOOR}  |  Buffer Threshold: {BUFFER}")
print(f"PortfolioConstructor.select() verdict: {len(held_df[held_df['Pct Rank'] < EXIT_FLOOR])} evicted, all {len(held_df)} retained\n")

display(held_df.style
    .set_caption("Current Holdings — Composite Percentile Rank (source: features_latest.parquet)")
    .background_gradient(cmap="RdYlGn", subset=["Pct Rank", "Composite Z"])
    .format({"Composite Z": "{:+.4f}", "Pct Rank": "{:.4f}"}))

# ── Top 20 by composite ────────────────────────────────────
print("\nTop 20 by Composite Z-Score:")
top20 = latest[["composite", "pct_rank"]].head(20).copy()
top20["Status"] = top20.index.map(lambda s: "*HOLD*" if s in current_holdings else "new")
top20.columns = ["Composite Z", "Pct Rank", "Status"]
display(top20.style
    .set_caption("Top 20 Symbols by Composite Z-Score")
    .background_gradient(cmap="Greens", subset=["Composite Z", "Pct Rank"])
    .format({"Composite Z": "{:+.4f}", "Pct Rank": "{:.4f}"}))

## 2. สามตัวเลือกสำหรับ June Rebalance

| Option | หลักการ | จำนวนหุ้น | Turnover |
|--------|---------|-----------|----------|
| **A — Systematic** | ถือ 9 หุ้นเดิมตามกฎ backtest (exit floor 0.35 → ไม่มีใครต่ำกว่า) | 9 | 0% |
| **B — Discretionary** | ตัด IRPC/PTTGC/JTS จาก drawdown (ไม่ใช่สัญญาณ) — การตัดสินใจนอกโมเดล | 9 | ~33% |
| **C — Backtest Top-10 ✅** | 10 อันดับแรกโดย composite z-score จากโมเดล — **สอดคล้องกับ backtest `N_CONCENTRATED=10`** | 10 | ~30–40% |

> **Option C คือทางเลือกที่สอดคล้องกับแบบจำลองมากที่สุด** — หมุนหุ้นตามสัญญาณ composite ไม่ใช่ตามราคา

In [ ]:
# ── Build all three option slates ──────────────────────────

# Option A: all 9 holdings (no rotation)
opt_a = held_df["Symbol"].tolist()

# Option B: discretionary 3-out/3-in (from May report daily-log plan)
opt_b_sell = ["IRPC", "PTTGC", "JTS"]
opt_b_buy = ["KCE", "EASTW", "KKP"]
opt_b = [s for s in current_holdings if s not in opt_b_sell] + opt_b_buy

# Option C: top 10 by composite z-score (the model-aligned choice)
opt_c = composite.nlargest(10).index.tolist()
opt_c_sell = [s for s in current_holdings if s not in opt_c]
opt_c_buy = [s for s in opt_c if s not in current_holdings]

# ── Display the three options side-by-side ─────────────────
max_len = max(len(opt_a), len(opt_b), len(opt_c))
comparison = pd.DataFrame({
    "Slot": [f"{i+1}" for i in range(max_len)],
    "A: Systematic (hold-9)": opt_a + [""] * (max_len - len(opt_a)),
    "B: Discretionary (3-out/3-in)": opt_b + [""] * (max_len - len(opt_b)),
    "C: Backtest Top-10 ✅": opt_c + [""] * (max_len - len(opt_c)),
})

print("=== THREE JUNE BOOKS — COMPARISON ===\n")
display(comparison.style.set_caption("June 2 ATO — Three Options (data from model)"))

# Rotation summary
print(f"\nOption C rotation:")
print(f"  SELL ({len(opt_c_sell)}): {', '.join(opt_c_sell)}")
print(f"  BUY  ({len(opt_c_buy)}): {', '.join(opt_c_buy)}")
print(f"  KEEP ({len(opt_c) - len(opt_c_buy)}): {', '.join(s for s in opt_c if s in current_holdings)}")
print(f"  Turnover: ~{len(opt_c_sell) / len(opt_c) * 100:.0f}% one-way")

## 3. ทำไม Option C — ข้อมูลจากโมเดล

### 3.1 Option C ตัดหุ้นที่สัญญาณอ่อน — ไม่ใช่หุ้นที่ขาดทุน

นี่คือข้อแตกต่างที่สำคัญที่สุดระหว่าง B (discretionary) และ C (model-driven):

- **B ตัด IRPC (−21%), PTTGC (−18.43%), JTS (−8.24%)** — เพราะราคาลง
- **C ตัด NEX (rank #68), AGE (#59), JTS (#62)** — เพราะ composite z-score ต่ำ

**IRPC และ PTTGC ยังอยู่อันดับ #5 และ #7 จาก 132 หุ้น** — composite z-score สูงมากเพราะ momentum 12m/6m/3m ข้ามเดือนล่าสุด (May drawdown ยังไม่เข้า signal) การขายเพราะราคาลง 1 เดือนคือการ override systematic edge ด้วยอารมณ์

In [ ]:
# ── Why C: compare composite rank vs price return ──────────
why_c_data = []
for sym in current_holdings:
    rank_pos = (composite > latest.loc[sym, "composite"]).sum() + 1
    in_b = "SELL" if sym in opt_b_sell else "KEEP"
    in_c = "SELL" if sym in opt_c_sell else "KEEP"
    why_c_data.append({
        "Symbol": sym,
        "Composite Z": latest.loc[sym, "composite"],
        "Pct Rank": latest.loc[sym, "pct_rank"],
        "Rank": rank_pos,
        "B Action": in_b,
        "C Action": in_c,
    })

# Add new buys for C
for sym in opt_c_buy:
    rank_pos = (composite > latest.loc[sym, "composite"]).sum() + 1
    why_c_data.append({
        "Symbol": f"→ {sym}",
        "Composite Z": latest.loc[sym, "composite"],
        "Pct Rank": latest.loc[sym, "pct_rank"],
        "Rank": rank_pos,
        "B Action": "BUY" if sym in opt_b_buy else "—",
        "C Action": "BUY",
    })

why_c_df = pd.DataFrame(why_c_data).sort_values("Pct Rank", ascending=False)
why_c_df = why_c_df.reset_index(drop=True)

display(why_c_df.style
    .set_caption("Why C: Rank-Driven Rotation (not price-driven)")
    .background_gradient(cmap="RdYlGn", subset=["Composite Z", "Pct Rank"])
    .format({"Composite Z": "{:+.4f}", "Pct Rank": "{:.4f}"}))

### 3.2 Composite Z-Score: Top 10 vs Current 9 vs Exit Floor

กราฟด้านล่างแสดง composite z-score ของหุ้นทั้ง 132 ตัว — จุดสีเขียวคือ Option C top-10, จุดสีส้มคือหุ้นที่ Option C จะขาย (NEX, AGE, JTS) เส้นประแดงคือ Exit Floor (35th percentile)

In [ ]:
# ── Composite Z-Score Bar Chart: Top 10 vs Current ─────────
fig, ax = plt.subplots(figsize=(14, 7))

# All symbols sorted by composite
all_sorted = latest.sort_values("composite", ascending=True)
colors_all = []
for sym in all_sorted.index:
    if sym in opt_c:
        colors_all.append("#1a9850")       # Top-10 = green
    elif sym in opt_c_sell:
        colors_all.append("#fdae61")       # Being sold = orange
    elif sym in current_holdings:
        colors_all.append("#4575b4")       # Other holdings = blue
    else:
        colors_all.append("#e0e0e0")       # Rest = grey

bars = ax.barh(range(len(all_sorted)), all_sorted["composite"], 
               color=colors_all, edgecolor="white", height=0.8, linewidth=0.3)

# Annotate held symbols
for i, (sym, row) in enumerate(all_sorted.iterrows()):
    if sym in current_holdings or sym in opt_c_buy:
        label = f"{sym} (#{(composite > row['composite']).sum() + 1})"
        ax.text(row["composite"] + 0.05, i, label, va="center", fontsize=8,
               fontweight="bold", color="#333333")

# Exit floor line
exit_floor_val = composite.quantile(EXIT_FLOOR)
ax.axvline(exit_floor_val, color="#d73027", linewidth=1.5, linestyle="--", alpha=0.7,
          label=f"Exit Floor (35th pct) = {exit_floor_val:+.2f}")

# Legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#1a9850", markersize=10, label="Top-10 (Option C)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#4575b4", markersize=10, label="Held (kept)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#fdae61", markersize=10, label="Option C sells (NEX/AGE/JTS)"),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#e0e0e0", markersize=10, label="Rest of universe"),
    Line2D([0], [0], color="#d73027", linewidth=1.5, linestyle="--", label=f"Exit Floor ({EXIT_FLOOR*100:.0f}th pct)"),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9, ncol=2)

ax.set_xlabel("Composite Z-Score (mean of 4 momentum signals)")
ax.set_title("Cross-Sectional Composite Z-Score — All 132 Symbols", fontweight="bold")
ax.set_yticks([])
sns.despine(left=True)
plt.tight_layout()
plt.show()

print(f"\nExit Floor (35th pct) composite value: {exit_floor_val:+.4f}")
print(f"All 9 holdings are above the exit floor → 0 unconditional evictions")
print(f"Lowest held: NEX at composite={latest.loc['NEX', 'composite']:+.4f} (pct_rank={latest.loc['NEX', 'pct_rank']:.4f})")
print(f"  → Distance above floor: {latest.loc['NEX', 'composite'] - exit_floor_val:+.4f} z-score")

### 3.3 Sector Diversification — Option C มี diversification ดีกว่า A

Option C เพิ่มจำนวนหุ้นจาก 9 → 10 (10% ต่อตัว) และกระจาย sector ได้ดี:

In [ ]:
# ── Sector mapping (from live-test groupings used in May report) ──
# NOTE: Confirm against SET-official classifications before locking orders.
SECTORS = {
    "DELTA": "Electronic Components",
    "HANA": "Electronic Components",
    "KCE": "Electronic Components",
    "GUNKUL": "Energy & Utilities",
    "IRPC": "Energy & Utilities",
    "EASTW": "Energy & Utilities",
    "PTTGC": "Petrochemicals",
    "INSET": "ICT",
    "MCOT": "ICT",
    "EPG": "Automotive",
    "NEX": "Automotive",
    "AGE": "Automotive",
    "JTS": "Financials",
    "KKP": "Financials",
    "LHFG": "Financials",
}

# Option C sector breakdown
opt_c_sectors = {}
for sym in opt_c:
    sec = SECTORS.get(sym, "Unknown")
    opt_c_sectors[sec] = opt_c_sectors.get(sec, 0) + 1

print("=== Option C — Sector Breakdown (10% each) ===")
sector_summary = []
for sec, count in sorted(opt_c_sectors.items(), key=lambda x: -x[1]):
    pct = count * 10
    status = "✓" if pct <= 35 else "⚠️ OVER CAP"
    sector_summary.append({"Sector": sec, "Count": count, "Weight": f"{pct}%", "Cap Check": status})

sector_df = pd.DataFrame(sector_summary)
display(sector_df.style.set_caption("Option C Sector Exposure — Max 35% Cap"))

print(f"\nMax sector concentration: {max(opt_c_sectors.values()) * 10}%")
print(f"All sectors {'✓ within' if max(opt_c_sectors.values()) * 10 <= 35 else '⚠️ EXCEEDS'} 35% cap")

### 3.4 สรุป: ทำไม Option C ถึงถูกต้องตามโมเดล

| ปัจจัย | A (Systematic) | B (Discretionary) | C (Backtest Top-10) |
|--------|:---:|:---:|:---:|
| **สอดคล้องกับ backtest** | ✅ หยุดหมุน | ❌ override โมเดลด้วยราคา | ✅ `N_CONCENTRATED=10` |
| **จำนวนหุ้น** | 9 | 9 | **10** (diversified กว่า) |
| **ตัดสินใจจาก** | Exit floor (ไม่มีใครต่ำกว่า 0.35) | Drawdown (อารมณ์) | **Composite z-score (สัญญาณ)** |
| **Sector diversification** | Energy 32% / Elec 22% / ICT 22% | คล้าย A + เพิ่ม KCE (เสี่ยง Elec cap) | **30/30/20/10/10 — กระจายดีกว่า** |
| **ความเสี่ยง** | ถือ IRPC/PTTGC ที่ −20% โดยไม่มี hedge | พลาด rebound ถ้าสัญญาณถูก | **หมุนตามสัญญาณ — โมเดล self-corrects เดือนถัดไป** |

**Bottom line:** Option C เชื่อใน systematic edge — ขายหุ้นที่สัญญาณอ่อน (NEX, AGE, JTS) แม้บางตัวกำไร และถือหุ้นที่สัญญาณแข็ง (IRPC, PTTGC) แม้ขาดทุน นี่คือสิ่งที่ backtest ทำมา 17 ปี และเป็นวิธีเดียวที่ edge จะทำงานในระยะยาว

## 4. Option C — พอร์ต June (10 หุ้น)

พอร์ตสุดท้ายคือ 10 อันดับแรกโดย composite z-score จาก features_latest.parquet (refresh 1 มิ.ย. 2026) equal-weight 10% ต่อตัว

In [ ]:
# ── Option C final portfolio table ─────────────────────────
# Last close prices from prices_latest.parquet
last_close = {}
for sym in opt_c:
    col = f"SET:{sym}"
    if col in prices.columns:
        last_close[sym] = float(prices[col].dropna().iloc[-1])
    else:
        last_close[sym] = np.nan

# Total NAV = sum of current positions MV + cash
total_nav = 1_003_342.71  # from May 29 close per monthly report
weight_per_position = total_nav / 10  # equal weight 10%

portfolio_c = []
for i, sym in enumerate(opt_c):
    held = "HOLD" if sym in current_holdings else "NEW"
    price = last_close.get(sym, np.nan)
    shares = int(weight_per_position / price / 100) * 100 if not np.isnan(price) else 0  # round to board lot
    mv = shares * price if not np.isnan(price) else 0
    portfolio_c.append({
        "#": i + 1,
        "Symbol": sym,
        "Composite Z": latest.loc[sym, "composite"],
        "Pct Rank": latest.loc[sym, "pct_rank"],
        "Sector": SECTORS.get(sym, "Unknown"),
        "Weight": "10%",
        "Indic. Shares": f"{shares:,}",
        "Indic. MV (THB)": f"{mv:,.0f}",
        "Status": held,
    })

port_df = pd.DataFrame(portfolio_c)
display(port_df.style
    .set_caption("Option C — June 2026 Portfolio (10 names, equal-weight 10% each)")
    .background_gradient(cmap="Greens", subset=["Composite Z", "Pct Rank"])
    .format({"Composite Z": "{:+.4f}", "Pct Rank": "{:.4f}"}))

# ── Trade List ─────────────────────────────────────────────
print("\n=== TRADE LIST (June 2 ATO) ===")
print()
print("SELL:")
for sym in opt_c_sell:
    held_info = next((p for p in live_config["positions"] if p["symbol"] == sym), None)
    if held_info:
        print(f"  {sym:8s}  {held_info['shares']:,} shares  @ avg_cost {held_info['avg_cost']:.4f}")
print()
print("BUY:")
for sym in opt_c_buy:
    price = last_close.get(sym, np.nan)
    shares = int(weight_per_position / price / 100) * 100 if not np.isnan(price) else 0
    print(f"  {sym:8s}  ~{shares:,} shares  @ ~{price:,.2f} THB (indicative)")
print()
print("After execution, record actual fills + commission-inclusive avg_cost in configs/live_portfolio.yaml")

## 5. Execution Plan — 2 มิ.ย. 2026 ATO

| Step | Action |
|------|--------|
| **Pre-market** | ตรวจสอบ SET-official sector classifications สำหรับ KCE, EASTW, MCOT, EPG |
| **Pre-market** | ตรวจสอบ ADTV ≥ 5M THB สำหรับ MCOT, EPG (หุ้นใหม่อันดับ 9-10) |
| **ATO (09:55–10:00)** | ส่งคำสั่ง SELL: NEX (99,000 หุ้น), AGE (80,600 หุ้น), JTS (1,900 หุ้น) |
| **ATO** | ส่งคำสั่ง BUY: KCE (~2,680 หุ้น), EASTW (~23,550 หุ้น), MCOT (~16,720 หุ้น), EPG (~17,300 หุ้น) |
| **Post-fill** | บันทึก avg_cost จากโบรกเกอร์ (commission-inclusive, 4 decimals) ลง `configs/live_portfolio.yaml` |
| **Post-fill** | อัพเดท cash balance ใน live_portfolio.yaml |
| **Same day** | Run daily refresh เพื่อ mark NAV ใหม่ |

> ⚠️ June 1 (Visakha Bucha substitution) และ June 3 (Queen Suthida's Birthday) เป็นวันหยุด SET — trading วันเดียวคือ June 2

## 6. Sign-off / Verification

ตรวจสอบความถูกต้องของข้อมูลก่อน execute จริง

In [ ]:
# ── Verification checks ────────────────────────────────────
print("=" * 60)
print("VERIFICATION — June 2026 Rebalance")
print("=" * 60)

checks = []

# 1. Data source
checks.append(("Features data source", "features_latest.parquet", "PASS"))

# 2. Latest date
latest_dt = pd.Timestamp(latest_date).strftime("%Y-%m-%d")
checks.append(("Latest feature date", latest_dt, "PASS" if "2026-05" in latest_dt else "WARN"))

# 3. Universe size
checks.append(("Universe size", f"{len(latest)} symbols", "PASS" if len(latest) >= 100 else "WARN"))

# 4. No holdings below exit floor
below_floor = held_df[held_df["Pct Rank"] < EXIT_FLOOR]
checks.append(("Holdings below exit floor", f"{len(below_floor)}", "PASS" if len(below_floor) == 0 else "FAIL"))

# 5. PortfolioConstructor.select() verdict
from csm.portfolio.construction import PortfolioConstructor, SelectionConfig
cross_section = latest[FEATURE_COLS].copy()
config = SelectionConfig(n_holdings_min=10, n_holdings_max=10)
pc = PortfolioConstructor()
result = pc.select(cross_section, current_holdings, config)
evicted_set = set(result.evicted)
expected_evictions = {"AGE", "NEX", "JTS"}
checks.append(("PortfolioConstructor evictions", f"{len(result.evicted)} ({', '.join(sorted(result.evicted))})",
    "PASS" if evicted_set == expected_evictions else "WARN"))

# 6. Option C = 10 symbols
checks.append(("Option C size", f"{len(opt_c)} symbols", "PASS" if len(opt_c) == 10 else "FAIL"))

# 7. Sector cap
max_sector_pct = max(opt_c_sectors.values()) * 10
checks.append(("Max sector exposure", f"{max_sector_pct}%", "PASS" if max_sector_pct <= 35 else "FAIL"))

# 8. Option C matches PortfolioConstructor output
pc_selected_set = set(result.selected)
opt_c_set = set(opt_c)
checks.append(("Option C matches PortfolioConstructor", "", "PASS" if pc_selected_set == opt_c_set else "WARN"))

# Display
check_df = pd.DataFrame(checks, columns=["Check", "Value", "Status"])

def _status_color(v):
    if v == "PASS":
        return "background-color: #1a9850; color: white"
    elif "WARN" in str(v):
        return "background-color: #fdae61"
    else:
        return "background-color: #d73027; color: white"

display(check_df.style.map(_status_color, subset=["Status"]))

all_pass = all(c[2] == "PASS" for c in checks)
print(f"\n{'✓ ALL CHECKS PASSED' if all_pass else '⚠️ SOME CHECKS NEED ATTENTION'}")

---

## Appendix: Data Provenance

| Data Source | File | Last Updated |
|-------------|------|---------------|
| Features (momentum signals) | `data/processed/features_latest.parquet` | 2026-06-01 11:01 BKK |
| Prices (daily OHLCV) | `data/processed/prices_latest.parquet` | 2026-06-01 11:01 BKK |
| Universe Symbols | `data/processed/universe_latest.parquet` | 2026-04-30 |
| Live Portfolio | `configs/live_portfolio.yaml` | Manual update |

**Strategy Version:** CSM v1.0 (live-test branch)
**Composite Formula:** `cross_section[['mom_12_1','mom_6_1','mom_3_1','mom_1_0']].mean(axis=1)` — exactly `PortfolioConstructor.select()`

---

*ทุกการตัดสินใจในสมุดบันทึกนี้คำนวณจากแบบจำลอง CSM โดยตรง — composite z-score, percentile rank, และ exit/buffer logic ล้วนมาจาก `PortfolioConstructor.select()` ใน `src/csm/portfolio/construction.py` ไม่มีการ override แบบ discretionary*

*Disclaimer: ผลตอบแทนในอดีตไม่ได้รับประกันผลตอบแทนในอนาคต การเทรดทั้งหมดอยู่ภายใต้ดุลยพินิจของผู้เทรด*